# NumPy – Quick Notes

A short reference: memory size of Python objects, array axes, ufunc reductions, slicing, boolean masks, fancy indexing, and list-comprehension patterns.

## 1. Python Objects and Memory

### Nested list comprehension
The outer comprehension builds the rows; the inner one builds the values in each row.

In [1]:
# 3 rows × 4 columns; each row starts 4 higher than the previous one
c = [[i + j for j in range(4)] for i in range(1, 10, 4)]
c

[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]

### Size of built-in objects
`sys.getsizeof()` returns the size of the object **itself** in bytes, not of anything it references. Even "empty" objects carry a fixed object-header overhead.

In [2]:
import sys

samples = {
    "int (0)": 0,
    "float (0.0)": 0.0,
    "bool (True)": True,
    "str ('')": "",
    "None": None,
    "object()": object(),
}

for name, value in samples.items():
    print(f"{name:<15} → {sys.getsizeof(value)} bytes")

int (0)         → 28 bytes
float (0.0)     → 24 bytes
bool (True)     → 28 bytes
str ('')        → 41 bytes
None            → 16 bytes
object()        → 16 bytes


In [3]:
# Strings grow by 1 byte per ASCII character (spaces included)
for elem in [3, True, None, '', ' ', 'qqq', 'q q q', 'qq qq']:
    print(f"{elem!r:<10} → {sys.getsizeof(elem)} bytes")

3          → 28 bytes
True       → 28 bytes
None       → 16 bytes
''         → 41 bytes
' '        → 42 bytes
'qqq'      → 44 bytes
'q q q'    → 46 bytes
'qq qq'    → 46 bytes


### NumPy scalar size by dtype
A NumPy scalar = a fixed object overhead + the raw value (`itemsize` bytes). Smaller dtypes save memory, which matters for large arrays (`arr.nbytes` = `arr.size × itemsize`).

In [4]:
import numpy as np

for dt in [np.int8, np.int16, np.int32, np.int64, np.float16, np.float32, np.float64]:
    print(f"{dt.__name__:<8} itemsize = {np.dtype(dt).itemsize} B    getsizeof(scalar) = {sys.getsizeof(dt(1))} B")

int8     itemsize = 1 B    getsizeof(scalar) = 25 B
int16    itemsize = 2 B    getsizeof(scalar) = 26 B
int32    itemsize = 4 B    getsizeof(scalar) = 28 B
int64    itemsize = 8 B    getsizeof(scalar) = 32 B
float16  itemsize = 2 B    getsizeof(scalar) = 26 B
float32  itemsize = 4 B    getsizeof(scalar) = 28 B
float64  itemsize = 8 B    getsizeof(scalar) = 32 B


### 🔍 Explicit vs Implicit

- **Explicit**: clearly stated or defined.
  _Example_: `x = 5` assigns a value directly.
- **Implicit**: implied, not directly stated.
  _Example_: Python implicitly converts `int` to `float` in mixed-type arithmetic (`1 + 2.0 → 3.0`).

### 📦 List vs Array

| | Python `list` | `array.array` | NumPy `ndarray` |
| --- | --- | --- | --- |
| **Element types** | Mixed | One type (homogeneous) | One dtype (homogeneous) |
| **Size** | Dynamic (`append`, `remove`) | Dynamic (`append`, `extend`) | Fixed — `np.append` returns a **new** array |
| **Vectorised maths** | ❌ | ❌ | ✅ |
| **Example** | `[1, "hello", True]` | `array.array('i', [1, 2, 3])` | `np.array([1, 2, 3])` |

🧠 **Tip:** use `list` for general-purpose collections; use NumPy arrays for numerical work where speed and type consistency matter.

## 2. Axes

- `axis=0` → operate **down the rows** (rows are collapsed) → one result **per column**
- `axis=1` → operate **across the columns** (columns are collapsed) → one result **per row**

In [5]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])

print("axis=0 (per column):", A.sum(axis=0))
print("axis=1 (per row):   ", A.sum(axis=1))

axis=0 (per column): [12 15 18]
axis=1 (per row):    [ 6 15 24]


## 3. Creating Random and Evenly Spaced Arrays

`np.random.seed()` is the legacy API; new code should prefer `rng = np.random.default_rng(42)` and `rng.integers(...)`.

In [6]:
np.random.seed(42)  # seed for reproducibility

x1 = np.random.randint(10, size=6)          # 1-D array
x2 = np.random.randint(10, size=(3, 4))     # 2-D array
x3 = np.random.randint(10, size=(3, 4, 5))  # 3-D array

print("ndim:", x1.ndim, x2.ndim, x3.ndim)
print("shape:", x1.shape, x2.shape, x3.shape)

ndim: 1 2 3
shape: (6,) (3, 4) (3, 4, 5)


In [7]:
b1 = np.random.randint(0, 1000, size=100)  # 100 random integers in [0, 1000)
b2 = np.linspace(0, 1000, 100)             # 100 evenly spaced floats from 0 to 1000 (inclusive)

print(b1[:10])
print(b2[:5])

[156  14 812  64 856 838 520 343 128 647]
[ 0.        10.1010101 20.2020202 30.3030303 40.4040404]


## 4. Slicing

`array[rows, cols]` — each part is `start:stop:step`; `stop` is excluded and a negative step walks backwards.

![Slicing examples on a 3×4 array](figures/img_index.png)

## 5. Ufunc `reduce` and `accumulate`

`reduce` collapses an array to a single value; `accumulate` keeps every intermediate result.

In [8]:
x = np.array([1, 2, 3, 4])

print(np.add.reduce(x))           # 1 + 2 + 3 + 4 = 10
print(np.add.accumulate(x))       # [ 1  3  6 10]
print(np.multiply.reduce(x))      # 1 * 2 * 3 * 4 = 24
print(np.multiply.accumulate(x))  # [ 1  2  6 24]

10
[ 1  3  6 10]
24
[ 1  2  6 24]


| ufunc method | Operation | NumPy shortcut | Pure-Python equivalent |
| --- | --- | --- | --- |
| `np.add.reduce` | Total sum | `np.sum(x)` | `functools.reduce(operator.add, x)` |
| `np.add.accumulate` | Running (cumulative) sum | `np.cumsum(x)` | `itertools.accumulate(x, operator.add)` |
| `np.multiply.reduce` | Total product | `np.prod(x)` | `functools.reduce(operator.mul, x)` |
| `np.multiply.accumulate` | Running (cumulative) product | `np.cumprod(x)` | `itertools.accumulate(x, operator.mul)` |

## 6. Boolean Arrays and Masks

![Boolean arrays and masks](figures/img_arrays_masks.png)

The same examples, runnable:

In [9]:
x = np.array([[1, 5, 3],
              [4, 2, 6]])

print(x < 5)                          # element-wise comparison → boolean array
print(np.sum(x < 5))                  # 4  (True counts as 1)
print(np.sum(x < 5, axis=1))          # [2 2]  per row
print(np.any(x < 5), np.all(x < 5))   # True False
print(np.sum((x > 2) & (x < 5)))      # 2  — use & / | (not and / or) and wrap each condition in ()

print(x[x < 5])                       # [1 3 4 2] — masking returns a 1-D array
print(x[x[:, 0] < 3])                 # rows whose first column is < 3 → [[1 5 3]]

[[ True False  True]
 [ True  True False]]
4
[2 2]
True False
2
[1 3 4 2]
[[1 5 3]]


## 7. Exercise: Indexing the Coloured Squares

![Revision – indexing exercise](figures/img_revision_indexing.png)

In [10]:
# 6×5 array holding 1..30
x = np.arange(1, 31).reshape(6, 5)
x

array([[ 1,  2,  3,  4,  5],
       [ 6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20],
       [21, 22, 23, 24, 25],
       [26, 27, 28, 29, 30]])

In [11]:
# 🟨 Yellow squares: rows 2–3, columns 0–1 (plain slicing)
x[2:4, 0:2]

array([[11, 12],
       [16, 17]])

### 🟦 Blue squares
Rows 0, 4 and 5, last two columns. Pick the rows with a list of indices (fancy indexing) or with a boolean mask, then slice the columns.

In [12]:
# Option 1: list of row indices
x[[0, 4, 5], 3:]

array([[ 4,  5],
       [24, 25],
       [29, 30]])

In [13]:
# Option 2: boolean mask that drops rows 1–3
row_mask = np.ones(x.shape[0], dtype=bool)
row_mask[1:4] = False
x[row_mask, 3:]

array([[ 4,  5],
       [24, 25],
       [29, 30]])

### 🟥 Red squares
The diagonal just above the main diagonal: positions (0,1), (1,2), (2,3), (3,4). Pass one array of row indices and one of column indices — NumPy pairs them element by element.

In [14]:
x[np.arange(4), np.arange(1, 5)]

array([ 2,  8, 14, 20])

In [15]:
# Equivalent: a 2-D boolean mask that is True where col == row + 1
mask = np.array([[j == i + 1 for j in range(x.shape[1])] for i in range(x.shape[0])])
x[mask]

array([ 2,  8, 14, 20])

## 8. Mixing Lists and Arrays

When a list meets an array in an arithmetic operation, NumPy converts the list to an array first, so `+` adds element-wise (for two plain lists, `+` would concatenate).

In [16]:
x1 = np.arange(10)
x2 = [-i - 1 if i % 2 == 1 else 0 for i in range(10)]  # odd i → -(i+1), even i → 0
print(x2)
print(x1 + x2)

[0, -2, 0, -4, 0, -6, 0, -8, 0, -10]
[ 0 -1  2 -1  4 -1  6 -1  8 -1]


In [17]:
# Chained conditional expression: three cases depending on i % 3
x4 = [-i - 1 if i % 3 == 2 else (3 * i if i % 3 == 1 else 0) for i in range(10)]
x4

[0, 3, -3, 0, 12, -6, 0, 21, -9, 0]

## 🔍 Two Roles of `if` in List Comprehensions

### 1. ✅ Filtering (`if` at the end)
**Includes or excludes** items from the result; the values themselves are unchanged.

```python
[i for i in range(6) if i < 1 or i > 3]   # [0, 4, 5]
```

### 2. 🎯 Conditional expression (`if … else` before `for`)
**Computes a different value** for each item; nothing is filtered out.

```python
[-i - 1 if i % 2 == 1 else 0 for i in range(10)]
```

### 📌 Summary

| Pattern | Purpose | Example |
| --- | --- | --- |
| `expr for item in iterable if cond` | **Filter items** | `[i for i in range(10) if i > 5]` |
| `expr1 if cond else expr2 for item in iterable` | **Choose a value** | `[i if i > 5 else 0 for i in range(10)]` |

🧠 **Tip:** use filtering to skip items; use a conditional expression to transform items by a rule. The filtering form can't have an `else`.